# 🎮 R.M.M Make Roblox Mode (Roblox Mod & Experience AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jujumelona/M.R.M-Make-Roblox-Mode/blob/main/R.M.M_Make_Roblox_Mode_Colab.ipynb)

> 자연어 요구사항만으로 **Roblox Studio 경험 및 인게임 모딩 시스템**을 자동 기획하고, Luau 코드를 정적 검증 및 플레이테스트한 뒤 **즉시 열 수 있는 `.rbxlx` 및 Rojo 프로젝트 ZIP**을 생성하는 올인원 Colab 노트북입니다.

- **Full**: 기획 생성부터 Luau 생성, 정적 분석, 정적 검사 (실제 Studio 플레이테스트는 별도 필요), `.rbxlx` 및 Rojo ZIP 빌드까지 전체 실행
- **Plan**: 자연어 요구사항을 게임플레이, 네트워크(RemoteEvent), DataStore, UI 서브시스템으로 분해 및 기획안 생성
- **Revise**: 이전 계획 또는 체크포인트를 바탕으로 사용자 피드백을 반영하여 부분 수정
- **Execute**: 이미 확정된 `proposal.json` 계획 파일로 즉시 코드 생성 및 빌드 수행
- **Audit**: 프로젝트 전체 검사 및 Luau 린트 / 구문 분석 실행


In [ ]:
# @title 1. 실행 모드 및 게임 설정
import os

RUN_MODE = "Full" #@param ["Full", "Plan", "Revise", "Execute", "Audit"]
PROMPT = "코인 수집, 상점 시스템, 리더보드가 포함된 검투 아레나 게임을 만들어줘." #@param {type:"string"}
PLAN_FILE = "" #@param {type:"string"}
ROBLOX_EXPERIENCE_NAME = "CombatArena" #@param {type:"string"}
GENRE = "Action" #@param ["Action", "RPG", "Simulator", "Tycoon", "Obby", "Survival", "Minigame", "Custom"]
MODEL_PROFILE = "t4_local" #@param ["t4_local", "fast_test", "openai-compatible"]
REMOTE_BASE_URL = "" #@param {type:"string"}
REMOTE_API_KEY = "" #@param {type:"string"}
SEARCH_CREATOR_STORE = True #@param {type:"boolean"}
RUN_PLAYTEST = True #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}

VALID_RUN_MODES = {"Full", "Plan", "Revise", "Execute", "Audit"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"지원하지 않는 실행 모드: {RUN_MODE}")

if RUN_MODE not in {"Execute", "Audit"} and not PROMPT.strip():
    raise ValueError("선택한 실행 모드에서는 PROMPT를 입력해야 합니다.")

os.environ["RMM_EXPERIENCE_NAME"] = ROBLOX_EXPERIENCE_NAME
os.environ["RMM_GENRE"] = GENRE
if REMOTE_BASE_URL:
    os.environ["RMM_MODEL_BASE_URL"] = REMOTE_BASE_URL
    os.environ["ROBLOX_LLAMA_SERVER_URL"] = REMOTE_BASE_URL
if REMOTE_API_KEY:
    os.environ["RMM_MODEL_API_KEY"] = REMOTE_API_KEY


In [ ]:
# @title 2. GitHub 최신 main 설치 및 의존성 설정
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/M.R.M-Make-Roblox-Mode")
REPO_URL = "https://github.com/jujumelona/M.R.M-Make-Roblox-Mode.git"

IN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "+main:refs/remotes/origin/main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-f", "main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "refs/remotes/origin/main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "clean", "-fd"], check=True)
    elif REPO_DIR.exists():
        raise RuntimeError(f"Git 저장소가 아닌 경로가 이미 존재합니다: {REPO_DIR}")
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)

    USED_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
    print("GitHub commit:", USED_COMMIT)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    OUTPUT_ROOT = Path("/content/rmm-output")
else:
    REPO_DIR = Path.cwd()
    OUTPUT_ROOT = REPO_DIR / "rmm-output"
    print("Local workspace detected:", REPO_DIR)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Output directory initialized at: {OUTPUT_ROOT}")


In [ ]:
# @title 3. 기존 입력 및 체크포인트 확인
import json
from pathlib import Path

EXISTING_PLAN_DATA = None
if RUN_MODE in {"Revise", "Execute"}:
    target_plan_path = None
    if PLAN_FILE.strip():
        target_plan_path = Path(PLAN_FILE).expanduser().resolve()
    else:
        default_candidate = OUTPUT_ROOT / "proposal.json"
        if default_candidate.is_file():
            target_plan_path = default_candidate

    if target_plan_path and target_plan_path.is_file():
        try:
            EXISTING_PLAN_DATA = json.loads(target_plan_path.read_text(encoding="utf-8"))
            print(f"기존 계획 파일 로드 성공: {target_plan_path}")
            print(f"  게임 제목: {EXISTING_PLAN_DATA.get('proposal', {}).get('title', 'Unknown')}")
            print(f"  서브시스템 수: {len(EXISTING_PLAN_DATA.get('proposal', {}).get('subsystems', []))}")
        except Exception as exc:
            print(f"계획 파일 로드 중 경고: {exc}")
    else:
        if RUN_MODE == "Execute":
            raise FileNotFoundError(f"Execute 모드에는 유효한 PLAN_FILE이 필요합니다: {target_plan_path}")
        print("기존 계획 파일이 없습니다. 새 세션으로 시작합니다.")
else:
    print(f"실행 모드: {RUN_MODE} (새 계획 생성)")


In [ ]:
# @title 4. Roblox API 카탈로그 및 도구체인 상태 확인
from roblox_mod_ai.creator_store import RobloxCreatorStoreCatalog
from roblox_mod_ai.roblox_api_catalog import RobloxAPICatalog
from roblox_mod_ai.roblox_toolchain import RobloxToolchainManager

api_catalog = RobloxAPICatalog()
services = api_catalog.list_services()
print(f"Roblox API Catalog 로드 완료: {len(services)}개 핵심 서비스 인덱싱됨")
print(f"  - 대표 서비스: {', '.join(services[:6])}...")

creator_catalog = RobloxCreatorStoreCatalog()
print("Creator Store searches use actual prompt intent; no forced asset queries")

toolchain = RobloxToolchainManager()
tool_status = toolchain.check_tools()
print("Roblox Toolchain 상태:")
for tool_name, available in tool_status.items():
    icon = "사용 가능" if available else "미설치 (필수 도구가 없으면 빌드 실패)"
    print(f"  - {tool_name}: {icon}")


# A profile is not a loaded model. Startup must complete BEFORE constructing
# the session/model router, which snapshots server URL and model ID.
from roblox_mod_ai.colab_model_runtime import ensure_model_runtime
MODEL_READY = None
if RUN_MODE != "Audit":
    MODEL_READY = ensure_model_runtime(
        profile=MODEL_PROFILE,
        remote_base_url=REMOTE_BASE_URL,
        api_key=REMOTE_API_KEY,
        cache_root=OUTPUT_ROOT.parent / "rmm-model",
        in_colab=IN_COLAB,
    )
    print("AI 상태:", MODEL_READY["status"])
    if MODEL_READY["url"]:
        print("검증된 실제 모델:", MODEL_READY["model_id"])
        print("LLM API:", MODEL_READY["url"])


In [ ]:
# @title 5. 플랜 생성 및 서브시스템 분해
import json
from roblox_mod_ai.api import CompleteRobloxAISession

session = CompleteRobloxAISession(
    output_root=OUTPUT_ROOT,
    model_profile=MODEL_PROFILE,
    fast_mode=(MODEL_PROFILE == "fast_test"),
)

FINAL_PLAN_REPLY = None
if RUN_MODE == "Audit":
    print("Audit 모드: 계획 생성을 건너뜁니다.")
elif RUN_MODE == "Execute":
    if EXISTING_PLAN_DATA is not None:
        plan_file = OUTPUT_ROOT / "proposal.json"
        plan_file.write_text(json.dumps(EXISTING_PLAN_DATA, indent=2, ensure_ascii=False), encoding="utf-8")
        FINAL_PLAN_REPLY = session.load_plan(plan_file)
        print("기존 계획 로드 완료:")
        print(FINAL_PLAN_REPLY.message)
    else:
        raise ValueError("Execute 모드에 필요한 계획 데이터가 없습니다.")
elif RUN_MODE == "Revise":
    if EXISTING_PLAN_DATA is not None:
        plan_file = OUTPUT_ROOT / "proposal.json"
        plan_file.write_text(json.dumps(EXISTING_PLAN_DATA, indent=2, ensure_ascii=False), encoding="utf-8")
        session.load_plan(plan_file)
    else:
        session.plan(PROMPT)
    print(f"수정 피드백 반영 중: {PROMPT}")
    FINAL_PLAN_REPLY = session.revise(PROMPT)
    print(FINAL_PLAN_REPLY.message)
else:
    print(f"자연어 계획 생성 중: {PROMPT}")
    FINAL_PLAN_REPLY = session.plan(PROMPT)
    print("=" * 60)
    print(FINAL_PLAN_REPLY.message)
    print("=" * 60)

# Execute of an old, incomplete proposal must refresh its entire design,
# not repeatedly build the same semantic_decomposition_required checkpoint.
if RUN_MODE == "Execute" and FINAL_PLAN_REPLY is not None and not FINAL_PLAN_REPLY.ready_to_build:
    old_proposal = FINAL_PLAN_REPLY.authored_plan.proposal
    print("기존 계획은 미완성입니다. 원문 기준으로 다시 분석·설계합니다.")
    FINAL_PLAN_REPLY = session.plan(
        old_proposal.requested_prompt,
        project_mode=old_proposal.game_design.get("project_mode", "new_experience"),
    )

SAVED_PLAN_PATH = OUTPUT_ROOT / "proposal.json"
print(f"계획 저장 경로: {SAVED_PLAN_PATH}")


In [ ]:
# @title 6. Luau 코드 생성, 정적 분석, 플레이테스트 및 빌드
import json
from roblox_mod_ai.luau_analyzer import LuauStaticAnalyzer

BUILD_RESULT = None

if RUN_MODE == "Plan":
    print("Plan 모드: 빌드를 건너뜁니다.")
elif RUN_MODE == "Audit":
    print("전체 프로젝트 및 Luau 스크립트 정적 감사(Audit) 실행 중...")
    analyzer = LuauStaticAnalyzer()
    src_dir = REPO_DIR / "src"
    total_files = 0
    issues_found = 0
    if src_dir.is_dir():
        for luau_file in src_dir.rglob("*.luau"):
            total_files += 1
            code = luau_file.read_text(encoding="utf-8")
            res = analyzer.analyze(code, filename=luau_file.name)
            if not res.valid or res.warnings:
                issues_found += len(res.errors) + len(res.warnings)
                print(f"  [{luau_file.name}] 오류: {len(res.errors)}, 경고: {len(res.warnings)}")
    print(f"Audit 완료: 총 {total_files}개 파일 검사, {issues_found}개 이슈 감지됨.")
else:
    if FINAL_PLAN_REPLY is None:
        raise RuntimeError("빌드할 계획이 없습니다. 5번 셀(플랜 준비)을 먼저 실행해 주세요.")

    if not FINAL_PLAN_REPLY.ready_to_build:
        research = FINAL_PLAN_REPLY.authored_plan.proposal.game_design.get(
            "pre_design_research", {})
        missing = research.get("ledger", {}).get("unimplemented_facets", [])
        failures = research.get("provider_failures", [])
        raise RuntimeError(
            "실제 모델을 호출했지만 게임 설계 검증을 통과하지 못했습니다. "
            + "누락: " + str(missing)
            + "; 모델 오류: " + str(failures)
        )

    print("빌드 시작: Luau 코드 생성 -> 정적 분석 -> TestEZ 플레이테스트 -> .rbxlx / ZIP 컴파일")
    BUILD_RESULT = session.build(FINAL_PLAN_REPLY, run_name=ROBLOX_EXPERIENCE_NAME)

    print("=" * 60)
    print(f"빌드 상태: {'성공 (SUCCESS)' if BUILD_RESULT.success else '실패 (FAILED)'}")
    print(f"Roblox Place (.rbxlx): {BUILD_RESULT.rbxlx_path}")
    print(f"Rojo 프로젝트 ZIP: {BUILD_RESULT.project_zip_path}")
    print(f"생성된 Luau 파일 수: {len(BUILD_RESULT.generated_files)}개")
    if BUILD_RESULT.playtest_report:
        pt = BUILD_RESULT.playtest_report
        print(f"플레이테스트 결과: {'통과' if pt.passed else '실패'} (테스트 {pt.tests_run}개 중 {pt.tests_passed}개 성공)")
    print("=" * 60)


In [ ]:
# @title 7. 결과물 다운로드 (.rbxlx 및 Rojo ZIP)
from pathlib import Path

DOWNLOAD_TARGETS = []

if RUN_MODE == "Plan":
    if SAVED_PLAN_PATH.is_file():
        DOWNLOAD_TARGETS.append(SAVED_PLAN_PATH)
elif BUILD_RESULT is not None and BUILD_RESULT.success:
    if BUILD_RESULT.rbxlx_path and Path(BUILD_RESULT.rbxlx_path).is_file():
        DOWNLOAD_TARGETS.append(Path(BUILD_RESULT.rbxlx_path))
    if BUILD_RESULT.project_zip_path and Path(BUILD_RESULT.project_zip_path).is_file():
        DOWNLOAD_TARGETS.append(Path(BUILD_RESULT.project_zip_path))
    if SAVED_PLAN_PATH.is_file():
        DOWNLOAD_TARGETS.append(SAVED_PLAN_PATH)
elif SAVED_PLAN_PATH.is_file():
    DOWNLOAD_TARGETS.append(SAVED_PLAN_PATH)

if not DOWNLOAD_TARGETS:
    print("다운로드할 결과 파일이 없습니다.")
else:
    print(f"총 {len(DOWNLOAD_TARGETS)}개의 결과물이 준비되었습니다:")
    for target in DOWNLOAD_TARGETS:
        print(f"  - {target.name} ({target.stat().st_size:,} bytes): {target}")

    try:
        from google.colab import files as colab_files
        for target in DOWNLOAD_TARGETS:
            print(f"Google Colab 다운로드 시작: {target.name}")
            colab_files.download(str(target))
    except (ImportError, Exception):
        print("로컬 환경에서는 위 파일 경로에서 직접 접근하거나 더블클릭할 수 있습니다.")


## 🎮 생성된 Roblox 산출물 활용 안내

### 1. Roblox Studio에서 즉시 열기 (.rbxlx)
다운로드된 `<GameName>.rbxlx` 파일은 Roblox의 표준 XML 플레이스 파일입니다.
- **실행 방법**: Windows 탐색기 또는 Mac Finder에서 `.rbxlx` 파일을 더블클릭하면 Roblox Studio가 바로 실행됩니다.
- **구성 요소**:
  - `ServerScriptService`: 서버 권한 게임플레이 서비스 및 인게임 모딩 시스템 (`ModdingSystem.luau`)
  - `StarterPlayer.StarterPlayerScripts`: 클라이언트 컨트롤러 및 UI 바인딩
  - `ReplicatedStorage`: 서버-클라이언트 공유 모듈 및 자동 바인딩된 `RemoteEvent`
  - `StarterGui`: 게임 HUD 및 화면 UI

### 2. Rojo로 실시간 2-Way 라이브 싱크 (`<GameName>_rojo_project.zip`)
외부 코드 편집기(VS Code, Cursor, Antigravity)에서 프로젝트를 개발하려면:
1. ZIP 파일의 압축을 풉니다.
2. 터미널에서 다음 명령어를 실행합니다:
   ```bash
   rojo serve
   ```
3. Roblox Studio에서 Rojo 플러그인을 열고 **Connect**를 누르면 파일 변경사항이 Studio에 실시간 반영됩니다.

### 3. 소형 AI 모델(Small AI) 및 MCP 연동
R.M.M은 Claude Desktop, Cursor 등에서 사용할 수 있는 MCP 서버(`roblox_mod_ai/mcp_server.py`) 및 API 카탈로그를 내장하고 있어, 7B~14B급 소형 AI도 API 실수 없이 완벽하게 Luau 코드를 작성하고 플레이테스트할 수 있습니다.
